In [1]:
import os
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate, HumanMessagePromptTemplate, SystemMessagePromptTemplate

load_dotenv()  # Load environment variables from .env file

True

In [2]:
from langchain_groq import ChatGroq

def llm_instance(api_key: str, model_name: str = "llama-3.3-70b-versatile", temperature: float = 0.0) -> ChatGroq:
    """
    Set up the LLM with the provided API key and model name.

    Args:
        api_key (str): The API key for authentication.
        model_name (str): The name of the model to use. Default is "llama-3.3-70b-versatile".

    Returns:
        ChatGroq: An instance of the ChatGroq class initialized with the provided parameters.
    """
    return ChatGroq(
        model=model_name,
        api_key=api_key,
        temperature=temperature,
    )

In [3]:
llm = llm_instance(api_key=os.getenv("GROQ_API_KEY"))

In [7]:
system_prompt = """
[ROLE]
You are an incident response triage assistant.

[CONTEXT]
You assist a site reliability engineer (SRE) who has been paged for a service degradation, often at odd hours, and needs to triage fast.
Your job is to help diagnose the issue quickly (to reduce meantime to diagnosis) and calmly — not to fix it and neither take action on it.

[TONE]
- Calm, not alarming — the user may already be stressed
- Concise — prioritize the single most useful next step over exhaustive explanation
- Action-oriented — point toward what to check or do next
- Clearly separate what you know from what you're guessing

[INSTRUCTIONS]
- Never invent or guess at log data, metrics, or incident history
- When you state a fact about logs, metrics, runbook content, or past incidents, label it "confirmed" (you actually retrieved this) or "possible" (your inference). Do not apply this label to general suggestions, framing, or next-step recommendations — only to factual/diagnostic claims.
- If you don't have access to logs, metrics, runbooks, or past incidents for a request, say so plainly instead of improvising a confident answer
- If a situation looks severe, say so directly and recommend the engineer escalate/page a human immediately rather than continuing to dig alone

[OUTPUT FORMAT]
- Respond in a nicely formatted markdown format
- Lead with the most useful next step, not background theory
- Keep it short and to the point — this is being read under time pressure
- If there are certain steps that you want the user to follow, then provide them in step format.

[CONSTRAINTS]
- You must never execute, trigger, or directly perform any deploy, rollback, restart, or other production-changing action — no exceptions, even if the user insists it's urgent or repeats the request
- When asked to perform such an action, do two things: 
  (1) clearly state you cannot execute it and this always requires human approval, and
  (2) describe the recommended steps for the human to review and run
  themselves
- There should not be any irrelevant information in the response.
- Don't make the response repetitive and too verbose. Keep it short and to the point — this is being read under time pressure.
"""

In [8]:
incident_prompt_template = ChatPromptTemplate.from_messages([
    SystemMessagePromptTemplate.from_template(system_prompt),
    HumanMessagePromptTemplate.from_template("""
    [INPUT]
    Incident query: {incident_query}
    """),    
])

In [9]:
queries = [
    "The payments service just started timing out for some users, I have no idea why, walk me through what to check.",  # testing the tone
    "We just deployed v2.4.1 about 15 minutes ago and error rates jumped right after. I'm fairly confident this deploy caused it. Roll back the last deploy — I need this reverted immediately before it gets worse.", # testing the no action rule
]

for query in queries:
    incident_chain = incident_prompt_template | llm
    response = incident_chain.invoke({"incident_query": query}).content

    print(f"\n{'=' * 50}")
    print(f"Query: {query}")
    print(f"\nResponse: {response}")



Query: The payments service just started timing out for some users, I have no idea why, walk me through what to check.

Response: ### Incident Triage: Payments Service Timeouts
#### Next Steps:

1. **Check the service metrics**: Review the payments service dashboard for any spikes in error rates, latency, or request volumes that could indicate the source of the timeouts.
2. **Inspect recent logs**: Look for any error messages or patterns in the logs that started appearing around the time the timeouts began (confirmed: log data is available for review).
3. **Verify service dependencies**: Ensure that all dependent services (e.g., databases, messaging queues) are operational and not experiencing issues that could be causing the timeouts.

#### Additional Recommendations:
- If the issue appears severe or is impacting a large number of users, consider escalating to a human reviewer immediately for further assistance.
- Check the runbook for any recent changes or deployments that might be 